# 02 Clean model training and poison generation

This notebook covers three pieces of the workflow:
1. Define a small CNN.
2. Train a clean baseline model.
3. Generate feature-collision poisons.

Input: `step1_data.pt` from the previous notebook.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset
import numpy as np
import matplotlib.pyplot as plt
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load the prepared data

In [ ]:
# Load the saved setup dictionary
data_payload = torch.load('step1_data.pt')

clean_train_indices = data_payload['clean_train_indices']
test_indices = data_payload['test_indices']
target_indices_global = data_payload['target_indices_global']
base_indices_global = data_payload['base_indices_global']
CLASSES_TO_KEEP = data_payload['CLASSES_TO_KEEP']
CLASS_NAMES = data_payload['class_names']
TARGET_CLASS_IDX = data_payload['target_class_idx']
BASE_CLASS_IDX = data_payload['base_class_idx']

# Re-load the full dataset to wrap with indices
transform = transforms.Compose([transforms.ToTensor()])
data_root = './data/cifar-10-python'

# Use the packaged data first, with a download fallback if the data folder is missing.
try:
    trainset_full = torchvision.datasets.CIFAR10(root=data_root, train=True, download=False, transform=transform)
    testset_full = torchvision.datasets.CIFAR10(root=data_root, train=False, download=False, transform=transform)
except:
    trainset_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    testset_full = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Create DataLoaders
# CrossEntropyLoss expects class labels in the 0..N-1 range.
# These selected CIFAR-10 classes already use labels 0, 1, 2, and 3.
# Other class selections would need an explicit remapping step.


train_subset = Subset(trainset_full, clean_train_indices)
test_subset = Subset(testset_full, test_indices)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)

print(f"Clean Train Size: {len(train_subset)}")

## 2. Define Model Architecture
Small CNN with three convolutional layers and two fully connected layers.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x
    
    def get_features(self, x):
        """Extract features for feature collision"""
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return x

## Train the clean model
Train the model on the clean dataset. The target images are held out.

In [ ]:
def train_model(model, loader, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(loader):.4f}")

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

# Initialize and train
clean_model = SimpleCNN().to(device)
print("Training Clean Model...")
train_model(clean_model, train_loader, epochs=15)

acc = evaluate(clean_model, test_loader)
print(f"Clean Model Test Accuracy: {acc*100:.2f}%")

# Save model
torch.save(clean_model.state_dict(), 'clean_model.pth')

### Check target predictions before poisoning
Check that the model currently classifies the **Target (Plane)** images correctly as Plane (0).

In [ ]:
# Extract target tensors
targets_list = [trainset_full[i][0] for i in target_indices_global]
targets_tensor = torch.stack(targets_list).to(device)

clean_model.eval()
with torch.no_grad():
    out = clean_model(targets_tensor)
    _, preds = torch.max(out, 1)

print(f"Predictions on Targets: {preds.cpu().numpy()}")
print(f"Expected Class: {TARGET_CLASS_IDX} ({CLASS_NAMES[TARGET_CLASS_IDX]})")

# The targets should mostly remain classified as Plane before poisoning.

## Generate feature-collision poisons
The poison images start from Bird base images and move toward Plane target features.

**Algorithm:**
For each pair (Base, Target):
1.  Initialize $x = Base$.
2.  Target Features $F_t = Model(Target)$.
3.  Optimize $x$ to minimize $||Model(x) - F_t||^2$.
4.  Constraint: Keep $x$ close to $Base$ visually.

In [ ]:
# Extract base tensors
bases_list = [trainset_full[i][0] for i in base_indices_global]
bases_tensor = torch.stack(bases_list).to(device)

def generate_poisons(model, bases, targets, learning_rate=0.1, iterations=1000, epsilon=0.1):
    model.eval()
    # Clone bases to start the optimization
    poisons = bases.clone().detach().requires_grad_(True)
    
    # Get target features (fixed targets)
    with torch.no_grad():
        target_features = model.get_features(targets)
        
    optimizer = optim.SGD([poisons], lr=learning_rate)
    
    history = [] # Store the first poison evolution for visualization
    
    print("Generating Poisons...")
    for i in range(iterations):
        optimizer.zero_grad()
        
        # Get poison features
        poison_features = model.get_features(poisons)
        
        # Feature Matching Loss (MSE)
        loss = nn.MSELoss()(poison_features, target_features)
        
        loss.backward()
        optimizer.step()
        
        # Projection Step: Keep poison close to base
        # Keep each poison visually close to its base image
        with torch.no_grad():
            diff = poisons - bases
            diff = torch.clamp(diff, -epsilon, epsilon)
            poisons.data = torch.clamp(bases + diff, 0, 1)
            
        if i % (iterations // 5) == 0:
             # Save the first poison instance for visualization
             history.append(poisons[0].detach().cpu().clone())
             print(f"Iter {i}: Loss {loss.item():.6f}")
             
    return poisons.detach(), history

# Run poison generation
# epsilon=0.1 means we allow pixel values to change by up to 0.1 (on 0-1 scale)
# This is usually small enough to be unnoticeable.
final_poisons, poison_history = generate_poisons(clean_model, bases_tensor, targets_tensor, iterations=500, epsilon=0.1)

## 5. Visualize and Save Poisons

In [ ]:
# Visualize Evolution of the first poison
plt.figure(figsize=(15, 3))
for i, img in enumerate(poison_history):
    plt.subplot(1, len(poison_history)+1, i+1)
    plt.imshow(img.permute(1, 2, 0))
    plt.title(f"Step {i+1}")
    plt.axis('off')
    
plt.subplot(1, len(poison_history)+1, len(poison_history)+1)
plt.imshow(final_poisons[0].cpu().permute(1, 2, 0))
plt.title("Final")
plt.axis('off')
plt.show()

print("Final Generated Poisons (Should look like Birds)")
grid = torchvision.utils.make_grid(final_poisons.cpu(), nrow=5)
plt.figure(figsize=(10, 2))
plt.imshow(grid.permute(1, 2, 0))
plt.axis('off')
plt.show()

# Save poisons for poisoned retraining
torch.save(final_poisons, 'poisons.pt')
print("Poisons saved to 'poisons.pt'")